# Figure S1. Fine-mapping methods at AD GWAS loci

Fit the benchmark methods at selected AD GWAS loci and record their running times.

- SuSiE-RSS (`susie_rss`) and SuSiE-rRSS (`susie_rrss`: `R_finite` = sketch dimension, `R_mismatch = "eb_mix"`, SER model when flagged), as in `09`
- DENTIST QC, DENTIST + RAISS, SLALOM + RAISS, CARMA, RSparsePro, with the settings of `06`

Run these two steps in order in the same job.

1. `[qc_inputs]` (one task per region): loads the sumstats (`getSumStatsDf` from `pecotmr_latest` via `load_all`) and the ADSP sketch LD as in the manual checks of `24`, and saves `<unit>.inputs.rds` (sumstats incl. z and positions, LD `R`, sketch dimension `B`, n). The locus plots use the same input files.
2. `[qc_methods]` (one task per region × method): a fresh R session reads the inputs; DENTIST / SLALOM / RAISS come from the installed `library(pecotmr)` (no `load_all`, which would replace it); SuSiE fits use the susieR GitHub build. `time_s` measures the method run time, including the time RSparsePro takes to write its $z$-score and LD input files. DENTIST and SLALOM runs also save the scores after QC (`z_qc`).

Regions (`--units`, `<study>.<block_id>`): Wightman full chr20_55439357_57610823; Wightman excl. 23andMe chr11_56858541_63818332 (MS4A, combined block); Wightman full chr11_84267999_86714492 (Fig S1 locus, PICALM); Bellenguez 2022 chr19_90949_1507536; EADB 2026 main chr16_81293081_82644764.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/result/<unit>/` (`<unit>.inputs.rds`, `<unit>.<method>.rds`); summary `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/summary/qc_methods_summary.{rds,tsv}` (CS counts, sizes, single-variant CSs, outliers, SER fallback, run time; plus the stored pipeline fits).

**Larger runs** (unit lists written by `[qc_units]`, section 0):
- every study × region whose SuSiE-RSS and SuSiE-rRSS credible sets differ (same rule as Fig 6B / S13B, APOE excluded), all methods → `ADGWAS_QC_methods/result/`, plotted by `AD_GWAS_QC_Methods_Loci`;
- the APOE block in every study, SuSiE-RSS and SuSiE-rRSS only → `ADGWAS_QC_methods_APOE/result/`, plotted by `FigED2.APOE_PIP_SuSiE_vs_rRSS` (SuSiE-RSS, SuSiE-rRSS before fallback, full pipeline).
- every SER fallback analysis outside APOE that is not already a differing region, all methods → `ADGWAS_QC_methods/result/` (`[qc_units_fallback]`, section 0b). A fallback analysis reports the SER model's single credible set; it is a differing region only when that set's variants differ from the SuSiE-RSS sets, so some fallback analyses were not in the differing list;
- the APOE block in every study with the other QC methods (DENTIST, DENTIST + RAISS, SLALOM + RAISS, CARMA, RSparsePro), added to `ADGWAS_QC_methods_APOE/result/` next to the SuSiE fits.


## 0. Units: differing regions and APOE

`[qc_units]` (interactive, a few minutes) writes the unit lists the submit cells read:
- `units_differ.txt` / `.tsv`: study × region pairs whose exported 95% credible sets differ between SuSiE-RSS and SuSiE-rRSS (credible sets compared by their variant sets, as in `Data_Processing/Figure_6/Fig6B_S13B`), APOE excluded;
- `units_apoe.txt`: every study with a SuSiE-rRSS fit of the APOE block.

In [ ]:
[qc_units]
parameter: fm_root = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10")
parameter: repo = path("/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper")
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/units_differ.txt', f'{cwd:a}/units_differ.tsv', f'{cwd:a}/units_apoe.txt'
R: expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container
    source(file.path(${repo:r}, "R", "utils.R"))          # read_gwas_tables(), apoe_block

    read_export <- function(sub, label)
      read_gwas_tables(file.path(${fm_root:r}, sub, "export")) %>%
        mutate(method = label) %>% filter(cs_coverage_0.95 != "susie_rss_0")
    result_full <- bind_rows(read_export("AD_GWAS_rss_ebmix_r4_L10",    "eb_mix"),
                             read_export("AD_GWAS_rss_baseline_r4_L10", "susie_rss"))

    ## per study x region x method: credible sets as sorted variant sets (index dropped), as in Fig 6B / S13B
    eff <- result_full %>%
      mutate(cs = as.character(cs_coverage_0.95)) %>% filter(!is.na(cs), cs != "0") %>%
      group_by(block_id, study, method, cs) %>%
      summarise(cs_sig = paste(sort(unique(variant_ID)), collapse = ","), .groups = "drop") %>%
      group_by(block_id, study, method) %>%
      summarise(n_eff = n(), sig = paste(sort(cs_sig), collapse = "|"), .groups = "drop")
    cmp <- eff %>%
      pivot_wider(names_from = method, values_from = c(n_eff, sig),
                  values_fill = list(sig = "", n_eff = 0L)) %>%
      mutate(different = coalesce(sig_eb_mix, "") != coalesce(sig_susie_rss, ""))

    differ <- cmp %>%
      filter(different, block_id != apoe_block) %>%
      transmute(unit = paste(study, block_id, sep = "."), study, block_id,
                n_cs_susie_rss = n_eff_susie_rss, n_cs_susie_rrss = n_eff_eb_mix) %>%
      arrange(block_id, study)
    cat(sprintf("Differing study x region pairs: %d in total, %d without APOE, in %d distinct regions\n",
                sum(cmp$different), nrow(differ), n_distinct(differ$block_id)))
    print(count(differ, n_cs_susie_rss, n_cs_susie_rrss))

    apoe <- list.files(file.path(${fm_root:r}, "AD_GWAS_rss_ebmix_r4_L10", "output", "fine_mapping"),
                       pattern = paste0("\\.", apoe_block, "\\.gwas_finemap\\.rds$")) %>%
      str_remove("\\.gwas_finemap\\.rds$") %>% sort()
    cat(sprintf("APOE: %d studies\n", length(apoe)))

    writeLines(differ$unit, ${_output[0]:r})
    write_tsv(differ, ${_output[1]:r})
    writeLines(apoe, ${_output[2]:r})

In [ ]:
# run interactively on a login node (a few minutes)
units_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/units"
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_units \
    --cwd ${units_dir}
wc -l ${units_dir}/units_differ.txt ${units_dir}/units_apoe.txt

## 0b. Units: SER fallback analyses

`[qc_units_fallback]` (interactive, seconds; after `[qc_units]`) reads the SER fallback flags of the final SuSiE-rRSS result (`export/SER_flag.tsv`, as Fig S12a) and writes:
- `units_fallback.tsv`: every flagged study × region, with whether it is the APOE block and whether it is already a differing region (`units_differ.tsv`);
- `units_fallback.txt`: the flagged units outside APOE that are not differing regions, i.e. the ones still to run. APOE is run separately (APOE submit cells in section 2).

In [ ]:
[qc_units_fallback]
parameter: fm_root = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10")
parameter: repo = path("/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper")
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/units_fallback.txt', f'{cwd:a}/units_fallback.tsv'
R: expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container
    source(file.path(${repo:r}, "R", "utils.R"))          # apoe_block

    differ_file <- file.path(${cwd:r}, "units_differ.tsv")
    if (!file.exists(differ_file)) stop("run [qc_units] first: ", differ_file, " not found")
    differ <- read_tsv(differ_file, show_col_types = FALSE)

    fb <- read_tsv(file.path(${fm_root:r}, "export", "SER_flag.tsv"), show_col_types = FALSE) %>%
      distinct(study, block_id, .keep_all = TRUE) %>%
      filter(serFallbackUsed %in% TRUE) %>%
      transmute(unit = paste(study, block_id, sep = "."), study, block_id,
                apoe = block_id == apoe_block, in_differ = unit %in% differ$unit) %>%
      arrange(apoe, block_id, study)

    cat(sprintf("SER fallback analyses: %d (APOE %d); outside APOE %d, of which %d already differing regions, %d to run\n",
                nrow(fb), sum(fb$apoe), sum(!fb$apoe), sum(!fb$apoe & fb$in_differ), sum(!fb$apoe & !fb$in_differ)))
    print(as.data.frame(fb))

    todo <- filter(fb, !apoe, !in_differ)
    writeLines(todo$unit, ${_output[0]:r})
    write_tsv(fb, ${_output[1]:r})

In [ ]:
# run interactively on a login node (seconds), after [qc_units]
units_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/units"
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_units_fallback \
    --cwd ${units_dir}
wc -l ${units_dir}/units_fallback.txt

## 1. Inputs

In [ ]:
[qc_inputs]
# One task per study x region: load the AD GWAS sumstats (needs pecotmr_latest via load_all) and the ADSP sketch LD,
# as in the manual checks of 24, and save them. The method step reads this file in a fresh session; the saved z and
# variant positions are also the input for plotting tracks.
parameter: units = ["AD_Wightman_Full_2021.chr20_55439357_57610823",
                    "AD_Wightman_Excluding23andMe_2021.chr11_56858541_63818332",
                    "AD_Wightman_Full_2021.chr11_84267999_86714492",
                    "AD_Bellenguez_2022.chr19_90949_1507536",
                    "AD_Bellenguez_main_2026.chr16_81293081_82644764"]
# <study>.<block_id>; chr11_84267999_86714492 is the Fig S1 locus (PICALM)
parameter: fm_root = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10")
parameter: ld_meta_file = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "10h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: for_each = "units"
output: f'{cwd:a}/{_units}/{_units}.inputs.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_units}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(pecotmr)
    devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)

    unit     <- "${_units}"
    study    <- str_split_fixed(unit, "\\.", 2)[, 1]
    block_id <- str_split_fixed(unit, "\\.", 2)[, 2]
    blk      <- str_split_fixed(block_id, "_", 3)
    region   <- sprintf("%s:%s-%s", blk[, 1], blk[, 2], blk[, 3])

    dat     <- readRDS(file.path(${fm_root:r}, "output/sumstats", paste0(unit, ".gwas_sumstats.rds")))
    sumstat <- getSumStatsDf(dat)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    ld_meta_file <- ${ld_meta_file:r}
        res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
        res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
        sketch_ref           <- res_sketch$ref_panel

    ske = res_sketch$LD_matrix

    LD = crossprod(ske) / (nrow(ske) - 1)

    SNP <- sumstat$variant_id
    R   <- LD[SNP, SNP]

    inputs <- list(
      unit     = unit, study = study, block_id = block_id, region = region,
      sumstat  = as_tibble(sumstat) %>% select(any_of(c("variant_id", "chrom", "pos", "A1", "A2", "z", "beta", "se", "N"))),
      R        = R,                      # ADSP sketch LD, rows / columns in sumstat order
      B        = nrow(ske),              # sketch dimension (R_finite for SuSiE-rRSS)
      n        = median(sumstat$N, na.rm = TRUE)
    )
    saveRDS(inputs, ${_output:r}, compress = FALSE)
    cat(sprintf("%s: p = %d, n = %.0f, B = %d, max |z| = %.2f\n", unit, length(SNP), inputs$n, inputs$B, max(abs(sumstat$z))))

## 2. Methods

In [ ]:
[qc_methods]
# One task per (study x region, method), in a fresh R session that reads [qc_inputs]. DENTIST / SLALOM / RAISS come
# from the installed pecotmr (library(pecotmr), no load_all, nothing sourced from pecotmr) so their compiled routines
# are found; SuSiE fits use the susieR GitHub build. Settings as benchmarked in 06 (DENTIST, SLALOM + RAISS, CARMA,
# RSparsePro) and 09 (SuSiE-RSS, SuSiE-rRSS). Run time covers only the method, not reading the inputs.
parameter: units = ["AD_Wightman_Full_2021.chr20_55439357_57610823",
                    "AD_Wightman_Excluding23andMe_2021.chr11_56858541_63818332",
                    "AD_Wightman_Full_2021.chr11_84267999_86714492",
                    "AD_Bellenguez_2022.chr19_90949_1507536",
                    "AD_Bellenguez_main_2026.chr16_81293081_82644764"]
# <study>.<block_id>; chr11_84267999_86714492 is the Fig S1 locus (PICALM)
parameter: methods = ["susie_rss", "susie_rrss", "dentist_qc", "dentist_raiss", "slalom_raiss", "carma", "rsparsepro"]
parameter: rsparsepro_script = path("/sc/arion/projects/ad-omics/haochen/rss_qc/RSparsePro_LD/src/rsparsepro_ld.py")
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "100h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: for_each = ["units", "methods"]
output: f'{cwd:a}/{_units}/{_units}.{_methods}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_units}_{_methods}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(MASS)
    library(crayon)                 # used by the sourced susieR (combine_styles)
    library(Rfast)
    library(pecotmr)                # installed build: DENTIST / SLALOM / RAISS with their compiled routines

    # susieR GitHub version (SuSiE-RSS / SuSiE-rRSS with eb_mix)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    `%||%` <- function(a, b) if (is.null(a)) b else a
    stopifnot("refPanel" %in% names(formals(pecotmr::raiss)))   # installed build must have the camelCase raiss()

    method <- "${_methods}"
    inp    <- readRDS(file.path(${_output:dr}, "${_units}.inputs.rds"))
    unit <- inp$unit; study <- inp$study; block_id <- inp$block_id
    z    <- inp$sumstat$z
    SNP  <- inp$sumstat$variant_id
    R    <- inp$R; n <- inp$n; B <- inp$B
    cat("unit:", unit, " method:", method, " p =", length(z), "\n")

    ## ---- methods (06 / 09 settings) -----------------------------------------------
    ref_df <- inp$sumstat %>% select(variant_id, chrom, pos, A1, A2)

    run_raiss_on_outliers <- function(outlier_flags, z_vec, ref_df, LD_mat) {          # as in 06
      known_idx <- which(!outlier_flags)
      known_df <- data.frame(chrom = ref_df$chrom[known_idx], pos = ref_df$pos[known_idx],
        variant_id = ref_df$variant_id[known_idx], A1 = ref_df$A1[known_idx],
        A2 = ref_df$A2[known_idx], z = z_vec[known_idx], stringsAsFactors = FALSE)
      # installed pecotmr (camelCase API since the June 2026 refactor): same settings as 06
      result <- pecotmr::raiss(refPanel = ref_df, knownZscores = known_df, ldMatrix = LD_mat,
                               lamb = 0.01, rcond = 0.01, r2Threshold = 0.6, minimumLd = 5, verbose = FALSE)
      combined_z <- z_vec
      raiss_df <- result$resultNofilter %||% result$result_nofilter
      if (!is.null(raiss_df)) {
        for (i in which(outlier_flags)) {
          vid <- ref_df$variant_id[i]; ri <- which(raiss_df$variant_id == vid)
          if (length(ri) == 1 && !is.na(raiss_df$z[ri])) combined_z[i] <- raiss_df$z[ri]
        }
      }
      combined_z
    }
    susie_qc <- function(z_vec) susie_rss(z = z_vec, R = R, n = n, control = susie_rss_control(check_prior = FALSE))

    run_method <- function(method) {
      switch(method,
        susie_rss  = list(fit = susie_rss(z = z, R = R, n = n)),
        susie_rrss = {
          fit  <- susie_rss(z = z, R = R, n = n, R_finite = B, R_mismatch = "eb_mix")
          flag <- isTRUE(fit$R_finite_diagnostics$R_reliability_flag)
          ## full pipeline: the SER model replaces the fit when the reliability flag is set
          list(fit = if (flag) fit$R_finite_diagnostics$ser_model else fit, full_fit = fit, ser_fallback = flag)
        },
        dentist_qc = {
          sw <- pecotmr::dentistSingleWindow(zScore = z, R = R, nSample = n)
          z_qc <- z; z_qc[sw$outlier] <- sw$imputed_z[sw$outlier]
          list(fit = susie_qc(z_qc), n_outliers = sum(sw$outlier), z_qc = z_qc)
        },
        dentist_raiss = {
          sw <- pecotmr::dentistSingleWindow(zScore = z, R = R, nSample = n)
          z_qc <- run_raiss_on_outliers(sw$outlier, z, ref_df, R)
          list(fit = susie_qc(z_qc), n_outliers = sum(sw$outlier), z_qc = z_qc)
        },
        slalom_raiss = {
          sl <- pecotmr::slalom(zScore = z, R = R)
          out <- sl$data$outliers; out[is.na(out)] <- FALSE
          z_qc <- run_raiss_on_outliers(out, z, ref_df, R)
          list(fit = susie_qc(z_qc), n_outliers = sum(out), z_qc = z_qc)
        },
        carma = {
          library(CARMA)
          cr <- CARMA(list(z), list(as.matrix(R)), lambda.list = list(1), outlier.switch = TRUE)
          cs <- cr[[1]]$`Credible set`[[2]]
          list(fit = list(sets = list(cs = if (length(cs)) cs else NULL), pip = cr[[1]]$PIPs, carma = cr))
        },
        rsparsepro = {
          pre <- file.path(tempdir(), unit)
          write_tsv(tibble(RSID = SNP, Z = z), paste0(pre, ".z.txt"))
          write.table(as.matrix(R), paste0(pre, ".ld.txt"), sep = " ", row.names = FALSE, col.names = FALSE, quote = FALSE)
          system(paste("python", shQuote(${rsparsepro_script:r}), "--z", shQuote(paste0(pre, ".z.txt")),
                       "--ld", shQuote(paste0(pre, ".ld.txt")), "--save", shQuote(pre)))
          rs <- read_tsv(paste0(pre, ".rsparsepro.txt"), show_col_types = FALSE) %>% mutate(index = row_number())
          k  <- setdiff(unique(rs$cs), 0)
          list(fit = list(sets = list(cs = if (length(k)) lapply(k, function(j) rs$index[rs$cs == j]) else NULL), rsparsepro = rs))
        },
        stop("unknown method: ", method))
    }

    ## ---- run + time (method only) ---------------------------------------------------
    t0  <- proc.time()[["elapsed"]]
    out <- tryCatch(run_method(method), error = function(e) list(error = conditionMessage(e)))
    time_s <- proc.time()[["elapsed"]] - t0

    cs  <- out$fit$sets$cs
    pip <- out$fit$pip
    output <- c(out, list(unit = unit, study = study, block_id = block_id, method = method,
                          time_s = time_s, n_snp = length(z), n = n, B = B,
                          cs_variants = if (length(cs)) lapply(cs, function(ix) SNP[ix]) else list(),
                          cs_lead     = if (length(cs)) vapply(cs, function(ix)
                                          SNP[ix][if (!is.null(pip)) which.max(pip[ix]) else 1], "") else character(0)))
    saveRDS(output, ${_output:r})
    cat(sprintf("[%s] %s: %d CS, %.1f s%s\n", unit, method, length(cs), time_s,
                if (!is.null(out$error)) paste0(" ERROR: ", out$error) else ""))

Submit (runs `qc_inputs`, then `qc_methods`)

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods"
job_dir="${base_dir}/${job_name}"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 144:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
# 1) inputs (load_all session), 2) methods (fresh sessions, installed pecotmr)
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_inputs \
    --cwd WORK_DIR/result \
    --mem 80G -J 4 && \
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_methods \
    --cwd WORK_DIR/result \
    --mem 80G -J 4
EOF

sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" ${job_name}.sh > ${job_name}.sh.submit
sb ${job_name}.sh.submit

Rerun only DENTIST + RAISS and SLALOM + RAISS (after the `raiss()` camelCase fix). The other methods' outputs are kept; `-s force` reruns these two although their earlier (error) `.rds` files exist.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods_raiss"
job_dir="${base_dir}/ADGWAS_QC_methods"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 24:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_methods \
    --methods dentist_raiss slalom_raiss \
    --cwd WORK_DIR/result \
    --mem 80G -J 4 -s force
EOF

sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" ${job_name}.sh > ${job_name}.sh.submit
sb ${job_name}.sh.submit

Submit: every differing study × region (APOE excluded), all methods, in chunks of 8 units per job. SoS skips the five Figure S1 regions when their outputs are current.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods_differ"
job_dir="${base_dir}/ADGWAS_QC_methods"
units_file="${job_dir}/units/units_differ.txt"          # from [qc_units]

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"
split -l 8 -d "${units_file}" differ_chunk_

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_CHUNK
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 144:00
#BSUB -o WORK_DIR/log/JOB_NAME_CHUNK.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_CHUNK.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb"
sos run "$nb" qc_inputs  --units $(cat WORK_DIR/code/CHUNK) --cwd WORK_DIR/result --mem 80G -J 4 && \
sos run "$nb" qc_methods --units $(cat WORK_DIR/code/CHUNK) --cwd WORK_DIR/result --mem 80G -J 4
EOF

for chunk in differ_chunk_*; do
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|CHUNK|${chunk}|g" ${job_name}.sh > ${job_name}_${chunk}.sh
  sb ${job_name}_${chunk}.sh
done

Submit: the APOE block (chr19:42,346,101–46,842,901, combined) in every study, SuSiE-RSS and SuSiE-rRSS only, in its own result folder. The block is large, so more memory per task.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods_APOE"
job_dir="${base_dir}/${job_name}"
units_file="${base_dir}/ADGWAS_QC_methods/units/units_apoe.txt"   # from [qc_units]

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=80000]
#BSUB -W 72:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb"
sos run "$nb" qc_inputs  --units $(cat UNITS_FILE) --cwd WORK_DIR/result --mem 150G -J 2 && \
sos run "$nb" qc_methods --units $(cat UNITS_FILE) --methods susie_rss susie_rrss --cwd WORK_DIR/result --mem 150G -J 2
EOF

sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|UNITS_FILE|${units_file}|g" ${job_name}.sh > ${job_name}.sh.submit
sb ${job_name}.sh.submit

Submit: the SER fallback analyses outside APOE that are not differing regions (`units_fallback.txt` from `[qc_units_fallback]`), all methods, into the same result folder as the differing regions, so `AD_GWAS_QC_Methods_Loci` picks them up.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods_fallback"
job_dir="${base_dir}/ADGWAS_QC_methods"
units_file="${job_dir}/units/units_fallback.txt"        # from [qc_units_fallback]

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"
if [ ! -s "${units_file}" ]; then echo "no fallback units to run"; else
split -l 8 -d "${units_file}" fallback_chunk_          # overwrites earlier chunk files of the same name

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_CHUNK
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 144:00
#BSUB -o WORK_DIR/log/JOB_NAME_CHUNK.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_CHUNK.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb"
sos run "$nb" qc_inputs  --units $(cat WORK_DIR/code/CHUNK) --cwd WORK_DIR/result --mem 80G -J 4 && \
sos run "$nb" qc_methods --units $(cat WORK_DIR/code/CHUNK) --cwd WORK_DIR/result --mem 80G -J 4
EOF

for chunk in fallback_chunk_*; do
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|CHUNK|${chunk}|g" ${job_name}.sh > ${job_name}_${chunk}.sh
  sb ${job_name}_${chunk}.sh
done
fi

Submit: the other QC methods at the APOE block in every study (`units_apoe.txt`), added to the APOE result folder, whose `<unit>.inputs.rds` already exist from the SuSiE-only run above (SoS skips the inputs step when its outputs are current). One job per study, since the combined block is large; CARMA and DENTIST can be slow here, so raise `-W` / `--walltime` if tasks time out.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADGWAS_QC_methods_APOE_qc"
job_dir="${base_dir}/ADGWAS_QC_methods_APOE"
units_file="${base_dir}/ADGWAS_QC_methods/units/units_apoe.txt"   # from [qc_units]

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_UNIT
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 144:00
#BSUB -o WORK_DIR/log/JOB_NAME_UNIT.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_UNIT.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb"
sos run "$nb" qc_inputs  --units UNIT --cwd WORK_DIR/result --mem 150G -J 1 && \
sos run "$nb" qc_methods --units UNIT --methods dentist_qc dentist_raiss slalom_raiss carma rsparsepro \
    --cwd WORK_DIR/result --mem 150G --walltime 140h -J 3
EOF

while read -r unit; do
  [ -n "${unit}" ] || continue
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|UNIT|${unit}|g" ${job_name}.sh > ${job_name}_${unit}.sh
  sb ${job_name}_${unit}.sh
done < "${units_file}"

## Summarize the completed runs

In [ ]:
[qc_methods_summary]
# One row per (study x region, method): CS counts and sizes, outliers, run time. Adds the stored pipeline fits
# (SuSiE-RSS baseline and SuSiE-rRSS) for comparison.
parameter: folder = path
parameter: fm_root = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10")
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/qc_methods_summary.rds', f'{cwd:a}/qc_methods_summary.tsv'
R: expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container
    library(tidyverse)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    cs_row <- function(cs_vars, lead = NULL) {
      sizes <- vapply(cs_vars, length, integer(1))
      tibble(n_cs = length(cs_vars), n_single_cs = sum(sizes == 1),
             cs_sizes = paste(sizes, collapse = ","),
             cs_lead  = paste(lead %||% vapply(cs_vars, function(v) v[1], ""), collapse = ";"))
    }

    files <- list.files(${folder:r}, pattern = "\\.rds$", recursive = TRUE, full.names = TRUE)
    files <- files[!grepl("\\.inputs\\.rds$", files)]                 # method outputs only
    if (!length(files)) stop("no method outputs in ", ${folder:r}, " yet")
    runs <- map_dfr(files, function(f) {
      r <- readRDS(f)
      bind_cols(tibble(study = r$study, block_id = r$block_id, method = r$method, time_s = r$time_s,
                       n_snp = r$n_snp, n_outliers = r$n_outliers %||% NA_integer_,
                       ser_fallback = r$ser_fallback %||% NA, error = r$error %||% NA_character_),
                cs_row(r$cs_variants %||% list(), r$cs_lead))
    })

    ## stored pipeline fits (fine_mapping/<unit>.gwas_finemap.rds), not timed here
    stored <- map_dfr(distinct(runs, study, block_id) %>% transpose(), function(u) {
      unit <- paste0(u$study, ".", u$block_id)
      map_dfr(c(susie_rss_pipeline = "AD_GWAS_rss_baseline_r4_L10", susie_rrss_pipeline = "AD_GWAS_rss_ebmix_r4_L10"), function(d) {
        f <- file.path(${fm_root:r}, d, "output/fine_mapping", paste0(unit, ".gwas_finemap.rds"))
        if (!file.exists(f)) return(NULL)
        fit <- attributes(readRDS(f))$elementMetadata$susieFit[[1]]
        snp <- names(fit$pip) %||% colnames(fit$alpha)
        cs   <- lapply(fit$sets$cs %||% list(), function(ix) if (!is.null(snp)) snp[ix] else as.character(ix))
        lead <- vapply(fit$sets$cs %||% list(), function(ix) {
                  k <- ix[which.max(fit$pip[ix])]; if (!is.null(snp)) snp[k] else as.character(k) }, "")
        bind_cols(tibble(study = u$study, block_id = u$block_id, method = NA_character_, time_s = NA_real_),
                  cs_row(cs, lead))
      }, .id = "stored") -> st
      if (!nrow(st)) return(NULL)
      st %>% mutate(method = stored) %>% select(-stored)
    })

    tb <- bind_rows(runs, stored) %>% arrange(study, block_id, method)
    print(as.data.frame(select(tb, study, block_id, method, n_cs, n_single_cs, cs_sizes, n_outliers, ser_fallback, time_s, error)))
    saveRDS(tb, ${_output[0]:r})
    write_tsv(tb, ${_output[1]:r})

In [ ]:
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_methods_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/result \
    --cwd /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/summary

# APOE block (SuSiE fits and, after the APOE QC run, the other methods)
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/29.FigS1.AD_GWAS_QC_Methods.ipynb" qc_methods_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods_APOE/result \
    --cwd /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods_APOE/summary
